# Evaluasi akhir Ciringoni M4

Proyek mendeteksi area kerusakan daun jagung dengan satu kelas `infected`. Notebook ini membaca hasil publik dan memeriksa bahwa baris test merujuk pada konfigurasi yang dikunci. Prosedur untuk mengulang training dan inferensi ada di [protokol](../PROTOKOL_FINAL.md).

Jalankan dari root clone atau folder `Data_Mining_Ciringoni_M1`. Jika memakai Kaggle dengan clone di direktori kerja lain, tetapkan variabel `CIRINGONI_REPO` ke root clone.

In [ ]:
import csv
import hashlib
import json
import os
from pathlib import Path

project = Path(os.environ.get('CIRINGONI_REPO', '.')).resolve()
if (project / 'Data_Mining_Ciringoni_M1').is_dir():
    project /= 'Data_Mining_Ciringoni_M1'

with (project / 'hasil_m2/evaluasi/eksperimen_final.csv').open(encoding='utf-8', newline='') as stream:
    rows = list(csv.DictReader(stream))

lock_file = project.parent / 'test-lock.json'
lock_text = lock_file.read_text(encoding='utf-8')
lock = json.loads(lock_text)
test, = (row for row in rows if row['split'] == 'test')
assert len(rows) == 7
assert all(row['images'] == '356' for row in rows if row['split'] == 'val')
assert test['images'] == str(lock['test']['images']) == '178'
assert test['size'] == str(lock['size']) and test['format'] == lock['precision']
assert test['lock_sha256'] == hashlib.sha256(lock_text.encode()).hexdigest()

cols = ['id', 'split', 'size', 'format', 'ap_small', 'ap_50_95', 'precision', 'recall', 'cpu_p95_ms']
header = '| ' + ' | '.join(cols) + ' |'
rule = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
data = ['| ' + ' | '.join(row[col] for col in cols) + ' |' for row in rows]
print('\n'.join([header, rule, *data]))

## Cara membaca hasil

Empat baris PyTorch membandingkan pasangan native 640 dan 1280 pada 5 dan 30 epoch. Dua baris OpenVINO membandingkan FP32 dan INT8 dari checkpoint terpilih pada 356 gambar validation. Kalibrasi INT8 menggunakan 320 gambar train. Baris test melaporkan satu konfigurasi yang dipilih dari validation.

AP menggunakan COCOeval dengan kotak di koordinat referensi 640. Presisi dan recall memakai confidence 0,05 dan IoU 0,50. Latensi CPU mencakup prapemrosesan, inferensi, pascapemrosesan, dan proyeksi balik kotak. Skor berasal dari satu seed dan satu koleksi citra.